In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Import necessary libraries
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('train.csv')

In [5]:
#use a dictionary to tell the computer the Level and Units for each column.
column_info = {
    'Loan_ID':           {'Level': 'Discrete', 'Units': '-'},
    'Gender':            {'Level': 'Nominal', 'Units': '-'},
    'Married':           {'Level': 'Nominal', 'Units': '-'},
    'Dependents':        {'Level': 'Ordinal', 'Units': 'Count'},
    'Education':         {'Level': 'Ordinal', 'Units': '-'},
    'Self_Employed':     {'Level': 'Nominal', 'Units': '-'},
    'ApplicantIncome':   {'Level': 'Discrete',   'Units': 'Currency ($)'},
    'CoapplicantIncome': {'Level': 'Continuous',   'Units': 'Currency ($)'},
    'LoanAmount':        {'Level': 'Continuous',   'Units': 'Currency (1000s)'},
    'Loan_Amount_Term':  {'Level': 'Continuous',   'Units': 'Months'},
    'Credit_History':    {'Level': 'Nominal', 'Units': 'Binary (1/0)'},
    'Property_Area':     {'Level': 'Nominal', 'Units': '-'},
    'Loan_Status':       {'Level': 'Nominal', 'Units': '-'}
}

In [6]:
table_rows = []

for column in df.columns:

    #grab the info for the current column from our dictionary above
    info = column_info[column]

    measure_level = info['Level']
    units = info['Units']
    unique_count = df[column].nunique()
    null_count = df[column].isnull().sum()

    #get data type (integer, float, or string)
    dtype = df[column].dtype
    if 'int' in str(dtype):
        formatted_dtype = "Integer"
    elif 'float' in str(dtype):
        formatted_dtype = "Float"
    else:
        formatted_dtype = "String (Text)"

    #check if number or string (pd.api.types.is_numeric_dtype(df[column]) checks if the column is a numerical data typre)
    if pd.api.types.is_numeric_dtype(df[column]) and (measure_level == 'Discrete' or measure_level == 'Continuous'):

        type_of_data = "Numerical"

        #RANGE (min - max)
        min_val = df[column].min()
        max_val = df[column].max()
        range_str = f"{min_val} - {max_val}"

        #top value for numbers is usually the max
        top_val = max_val

        #OUTLIERS (IQR Method)
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR

        #fix negative lower limits for money (e.g. Income can't be -1500)
        if lower_limit < 0:
            lower_limit = 0

        #count how many are outside the limits
        outlier_count = ((df[column] < lower_limit) | (df[column] > upper_limit)).sum()

        if outlier_count > 0:
            outliers_str = f"Yes, < {round(lower_limit, 1)} or > {round(upper_limit, 1)}"
        else:
            outliers_str = "No"

    #categorical (nominal numbers that are encoded like Credit History)
    else:
        type_of_data = "Categorical"
        min_val = "-"
        outliers_str = "No" # Categories don't have outliers

        #RANGE (List of Unique Values)
        #we get the unique values, convert to strin, and join them with commas
        unique_vals = df[column].dropna().unique()

        #convert all to string (e.g., 1.0 -> "1.0")
        unique_vals_str = [str(x) for x in unique_vals]

        #if there are too many (like > 3), just show top 3 + "etc"
        if len(unique_vals_str) > 3:
            range_str = ", ".join(unique_vals_str[:3]) + ", etc."
        else:
            range_str = ", ".join(unique_vals_str)

        #top value is the Mode
        if len(df[column].mode()) > 0:
            top_val = df[column].mode()[0]
        else:
            top_val = "N/A"

    #add to list
    table_rows.append({
        "Variable": column,
        "Type of Data": type_of_data,
        "Data Type": formatted_dtype,
        "Measurement Level": measure_level,
        "Units": units,
        "Range": range_str,
        "Min Value": min_val,
        "Top Value": top_val,
        "Unique Values": unique_count,
        "Null Values": null_count,
        "Outliers": outliers_str
    })

In [7]:
table1_df = pd.DataFrame(table_rows)

In [8]:
table1_df

,Variable,Type of Data,Data Type,Measurement Level,Units,Range,Min Value,Top Value,Unique Values,Null Values,Outliers
0,Loan_ID,Categorical,String (Text),Discrete,-,"LP001002, LP001003, LP001005, etc.",-,LP001002,614,0,No
1,Gender,Categorical,String (Text),Nominal,-,"Male, Female",-,Male,2,13,No
2,Married,Categorical,String (Text),Nominal,-,"No, Yes",-,Yes,2,3,No
3,Dependents,Categorical,String (Text),Ordinal,Count,"0, 1, 2, etc.",-,0,4,15,No
4,Education,Categorical,String (Text),Ordinal,-,"Graduate, Not Graduate",-,Graduate,2,0,No
5,Self_Employed,Categorical,String (Text),Nominal,-,"No, Yes",-,No,2,32,No
6,ApplicantIncome,Numerical,Integer,Discrete,Currency ($),150 - 81000,150,81000,505,0,"Yes, < 0 or > 10171.2"
7,CoapplicantIncome,Numerical,Float,Continuous,Currency ($),0.0 - 41667.0,0.0,41667.0,287,0,"Yes, < 0 or > 5743.1"
8,LoanAmount,Numerical,Float,Continuous,Currency (1000s),9.0 - 700.0,9.0,700.0,203,22,"Yes, < 0 or > 270.0"
9,Loan_Amount_Term,Numerical,Float,Continuous,Months,12.0 - 480.0,12.0,480.0,10,14,"Yes, < 360.0 or > 360.0"


In [9]:
table1_df.to_csv('table1.csv', index=False)